# <center>Laboratorio 9: Benchmark de Carga y Modelos con Spotify 🎵</center>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco e Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Bryan Cabezas 
- Nombre de alumno 2: Gonzalo Sobarzo

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

# Temas a tratar

- Lectura eficiente de datos en formato Parquet.
- Optimización del uso de memoria mediante conversión de tipos de datos.
- Paralelización de operaciones I/O con `ThreadPoolExecutor`.
- Comparación de implementaciones de predicción: Python, NumPy, Numba, pandas y Polars.
- Entrenamiento de modelos con RandomForestRegressor y efecto de `n_jobs`.
- Orquestación de pipelines de datos con Apache Airflow.

# Objetivos principales del laboratorio

- Cargar datos de canciones de Spotify desde archivos Parquet y optimizar su representación en memoria.
- Comparar el tiempo de lectura de archivos en serie vs. en paralelo.
- Analizar el impacto de distintas implementaciones (Python puro, NumPy, Numba, pandas, Polars) en el tiempo de predicción de un modelo lineal.
- Entrenar un RandomForestRegressor que prediga la valencia de canciones, comparando el efecto de la paralelización del entrenamiento.
- Orquestar el pipeline completo (carga + entrenamiento) usando Apache Airflow.

> Instalamos e importamos las librerías necesarias 🎸

In [1]:
!uv pip install pandas pyarrow lightgbm scikit-learn plotly apache-airflow polars numba

Using Python 3.14.3 environment at: /Users/bryan/Desktop/MDS/segundo semestre/Laboratorio de ciencia de datos/Repo/MDS7202/.venv
Resolved 140 packages in 3.47s                                       
⠙ Preparing packages... (0/80)                                                  
⠙ Preparing packages... (0/80)------------------     0 B/44.41 MiB           
⠙ Preparing packages... (0/80)------------------ 16.00 KiB/44.41 MiB         
⠙ Preparing packages... (0/80)------------------ 32.00 KiB/44.41 MiB         
⠙ Preparing packages... (0/80)------------------ 48.00 KiB/44.41 MiB         
⠙ Preparing packages... (0/80)------------------ 63.58 KiB/44.41 MiB         
⠙ Preparing packages... (0/80)------------------ 79.58 KiB/44.41 MiB         
⠙ Preparing packages... (0/80)------------------ 95.58 KiB/44.41 MiB         
⠙ Preparing packages... (0/80)------------------ 111.58 KiB/44.41 MiB        
⠙ Preparing packages... (0/80)------------------ 111.58 KiB/44.41 MiB        
tenacity          

In [2]:
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import numpy as np
import pandas as pd
import polars as pl
import numba
import plotly.express as px
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
import platform
import os

DATA_DIR = Path("data")

# 1. Carga y Optimización de Datos con Parquet

Los datos que usaremos en este laboratorio corresponden a un dataset de canciones de Spotify almacenado en **20 archivos Parquet** (`batch_01.parquet` … `batch_20.parquet`), con un total de 200 000 canciones y 24 columnas que incluyen características de audio, metadatos y la letra completa de cada canción.

A continuación trabajaremos en dos aspectos fundamentales de la carga de datos en la práctica:
1. **Optimizar el uso de memoria** ajustando los tipos de datos de las columnas.
2. **Reducir el tiempo de carga** paralelizando la lectura de archivos.

## 1.1 Exploración y Optimización de Tipos de Datos [1 Punto]

Cuando cargamos datos con pandas, los tipos inferidos por defecto no siempre son los más eficientes. Por ejemplo, un entero que siempre cabe en 16 bits se almacena por defecto como `int64` (64 bits), usando 4 veces más memoria de la necesaria. Lo mismo ocurre con flotantes y con columnas categóricas almacenadas como strings.

**Código dado** — funciones de carga:

In [4]:
def load_batch(path: str) -> pd.DataFrame:
    """Lee un único archivo Parquet y retorna un DataFrame."""
    return pd.read_parquet(path)


def load_all_serial(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en serie y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    return pd.concat([load_batch(str(p)) for p in paths], ignore_index=True)

**TO-DO [0.3 Puntos]:**
- [ ] Ejecutar `load_all_serial` sobre todos los batches y explorar el DataFrame resultante (`.dtypes`, `.memory_usage(deep=True)`).
- [ ] Aplicar las siguientes conversiones a un nuevo DataFrame (copia del originalmente cargado `df_opt`):
  - `float64` → `float32`: columnas de audio features (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `avg_artist_popularity`).
  - `int64` → `int16`: columnas `key`, `mode`.
  - `int64` → `int32`: columnas `year`, `popularity`, `duration_ms`, `total_artist_followers`.
- [ ] Comparar el uso de memoria antes y después con un gráfico de barras usando Plotly (código dado).

In [12]:
# Escribe aquí tu código
df = load_all_serial(DATA_DIR, n_batches=20)

df_opt = df.copy()  # .astype(...)

df_opt.dtypes

id                            str
name                          str
album_name                    str
artists                    object
danceability              float64
energy                    float64
key                         int64
loudness                  float64
mode                        int64
speechiness               float64
acousticness              float64
instrumentalness          float64
liveness                  float64
valence                   float64
tempo                     float64
duration_ms                 int64
lyrics                        str
year                        int64
genre                         str
popularity                  int64
total_artist_followers      int64
avg_artist_popularity     float64
artist_ids                 object
niche_genres               object
dtype: object

In [10]:
df_opt = df_opt.astype({
    "danceability": np.float32, "energy": np.float32, "loudness": np.float32,
    "speechiness": np.float32, "acousticness": np.float32, "instrumentalness": np.float32,
    "liveness": np.float32, "valence": np.float32, "tempo": np.float32,
    "avg_artist_popularity": np.float32,
    "key": np.int16, "mode": np.int16,
    "year": np.int32, "popularity": np.int32, "duration_ms": np.int32,
    "total_artist_followers": np.int32
})

# Compara el uso de memoria antes y después con un gráfico de barras
mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Antes", "Después"],
    y=[mem_before, mem_after],
    labels={"x": "Estado", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} MiB → {mem_after:.1f} MiB ({(1 - mem_after / mem_before) * 100:.1f}% reducción)",
).show()


### Preguntas [0.7 Puntos]

1. ¿Qué es el formato **Parquet**? ¿Qué ventajas tiene sobre CSV para datos analíticos? ¿Qué es *columnar storage* y por qué acelera las consultas que solo leen algunas columnas?
2. ¿Qué es **Apache Arrow**? ¿Cómo se relaciona con Parquet y con pandas internamente? ¿Qué ganas al usar `pd.read_parquet` en vez de `pd.read_csv`?
3. ¿Por qué existe `float32` si `float64` es más preciso? ¿En qué contextos esa pérdida de precisión es irrelevante?
4. ¿Cuándo **no** conviene reducir la precisión de un tipo numérico? ¿Qué riesgos concretos existen?
5. ¿Existe alguna alternativa a pandas para trabajar con estos datos de forma más eficiente en memoria? (menciona al menos dos)
6. ¿Cuánto se redujo el uso de memoria en total (en MiB y en %)? ¿Era esperable ese resultado? ¿Por qué no se redujo tanto como podría esperarse?
7. ¿Qué pasaría si intentaras reducir `valence` a `float16`? ¿Qué riesgo existiría para el modelo entrenado en la sección 2?

1. El formato Parquet es un formato de almacenamiento de datos columnar que permite una compresión eficiente y un acceso rápido a los datos. A diferencia de CSV, que almacena los datos en filas, Parquet almacena los datos por columnas, lo que permite leer solo las columnas necesarias para una consulta, reduciendo el tiempo de lectura y el uso de memoria. El almacenamiento columnar también facilita la compresión, ya que los valores en una columna suelen ser similares, lo que mejora la eficiencia de la compresión. El columnar storage acelera las consultas que solo leen algunas columnas porque solo se accede a los datos relevantes, evitando la necesidad de cargar toda la fila en memoria.
2. Apache Arrow es una plataforma de memoria compartida que proporciona un formato de datos en memoria eficiente y un conjunto de bibliotecas para trabajar con datos tabulares. Arrow se relaciona con Parquet porque Parquet puede usar Arrow como su formato de almacenamiento en memoria, lo que permite una lectura y escritura más rápida de los datos. Al usar `pd.read_parquet`, se aprovecha la eficiencia de Arrow para cargar los datos directamente en memoria, lo que resulta en tiempos de carga más rápidos en comparación con `pd.read_csv`, que no utiliza Arrow.
3. El tipo `float32` existe porque en muchos casos la precisión de `float64` no es necesaria, y el uso de `float32` puede reducir significativamente el uso de memoria. En contextos como el procesamiento de imágenes, audio o datos de sensores, donde la precisión extrema no es crucial, `float32` es suficiente y permite manejar conjuntos de datos más grandes sin agotar la memoria.
4. No conviene reducir la precisión de un tipo numérico cuando los datos requieren una alta precisión para ser útiles, como en cálculos científicos, financieros o cuando se trabaja con modelos de machine learning que son sensibles a pequeñas variaciones en los datos. El riesgo de reducir la precisión es que se pueden introducir errores de redondeo o pérdida de información, lo que puede afectar negativamente el rendimiento del modelo o la validez de los resultados.
5. Alternativas a pandas para trabajar con datos de forma más eficiente en memoria incluyen:
   - **Polars**: una biblioteca de procesamiento de datos en Rust(Lenguaje de programación que es mucho más rápido, ya que compila sin un recolector de basura) que ofrece un rendimiento superior al de pandas, especialmente en operaciones de lectura y escritura.
   - **Dask**: una biblioteca que permite el procesamiento de datos en paralelo y distribuidos, lo que es útil para manejar conjuntos de datos que no caben en memoria.
6. Se redujo solo un 3.6%, lo cuál es poco para el volumen de megabytes, pero para el volumen de datos es un resultado esperado. Esto se debe a que aunque se redujeron los tipos de datos, la cantidad total de datos sigue siendo la misma, y algunas columnas pueden no haber sido optimizadas o pueden contener valores que no se benefician tanto de la reducción de precisión.
7. Si intentaras reducir `valence` a `float16`, podrías enfrentar problemas de precisión, ya que `float16` tiene una menor capacidad para representar números con precisión en comparación con `float32`. Esto podría resultar en una pérdida significativa de información, especialmente si los valores de `valence` tienen una amplia gama o requieren una alta precisión para ser útiles en el modelo. El riesgo es que el modelo entrenado con datos de menor precisión podría tener un rendimiento significativamente peor, ya que no capturaría adecuadamente las variaciones en la valencia de las canciones.

In [14]:
# **IMPORTANTE**: Una vez contestada la pregunta, ejecutar esta celda para liberar memoria.
df_opt = None

## 1.2 Lectura en Serie vs. Paralelo [1 Punto]

Cuando se trabaja con múltiples archivos, la lectura **en paralelo** puede reducir el tiempo total al aprovechar que la espera de I/O (disco/red) no bloquea al procesador. En Python, la clase `ThreadPoolExecutor` del módulo `concurrent.futures` permite lanzar múltiples hilos para ejecutar operaciones de forma concurrente.

**TO-DO: [0.3 Puntos]**
- [ ] Implementar `load_all_parallel` usando `ThreadPoolExecutor`.
- [ ] Medir con `%timeit` ambas versiones sobre todos los batches.
- [ ] Generar un gráfico de línea (Plotly) con los tiempos para 2, 4, 6, …, 20 archivos, con series `Serial` y `Paralelo`.

In [5]:
# Escribe aquí tu código
def load_all_parallel(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en paralelo y los concatena usando ThreadPoolExecutor."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    with ThreadPoolExecutor() as executor:
        dfs = list(executor.map(lambda p: load_batch(str(p)), paths))
    return pd.concat(dfs, ignore_index=True)

**Benchmark:** mide tiempos para 2, 4, 6, ..., 20 archivos y grafica


In [19]:
# Le hacemos print a las características del sistema para entender mejor los resultados.
host_platform = getattr(numba.config, "HOST_PLATFORM", platform.platform())
cpu_name = getattr(numba.config, "CPU_NAME", None) or platform.processor() or "unknown"
num_threads = getattr(numba.config, "NUM_THREADS", None) or os.cpu_count() or "unknown"

print(f"Sistema: {host_platform}, CPU: {cpu_name}, Threads disponibles: {num_threads}")

Sistema: Windows-11-10.0.26200-SP0, CPU: AMD64 Family 23 Model 113 Stepping 0, AuthenticAMD, Threads disponibles: 12


In [15]:
@dataclass
class ReadMeasurement:
    n_files: int
    time_sec: float
    version: str


measurements: list[ReadMeasurement] = []


for n in range(2, 21):
    t0 = time.perf_counter()
    load_all_serial(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Serial"))

    t0 = time.perf_counter()
    load_all_parallel(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Paralelo"))

df_times = pd.DataFrame(measurements)
px.line(
    df_times,
    x="n_files",
    y="time_sec",
    color="version",
    markers=True,
    title="Tiempo de lectura: Serial vs Paralelo",
    labels={"n_files": "Número de archivos", "time_sec": "Tiempo (s)"},
).show()

### Preguntas  [0.7 Puntos]

1. ¿Qué significa que una operación sea **I/O-bound** vs **CPU-bound**? ¿A cuál categoría pertenece la lectura de archivos desde disco?
2. ¿Qué es el **GIL** (*Global Interpreter Lock*) de CPython? ¿Por qué existe? ¿Qué problema resuelve y qué limitación introduce?
3. ¿Por qué usamos Python si tiene el GIL? ¿Qué ganamos al usarlo como lenguaje de *pegamento* entre librerías de alto rendimiento (NumPy, Arrow, PyTorch…)?
4. ¿Cuándo conviene usar `ThreadPoolExecutor` vs `ProcessPoolExecutor`? ¿Cuál usarías si la operación fuera puramente CPU-bound?
5. ¿Qué overhead introduce crear un pool de threads? ¿Qué pasaría si los archivos fueran muy pequeños (p.ej. 1 KB cada uno)?
6. ¿Se observó mejora con la lectura paralela? ¿A partir de cuántos archivos empieza a ser notable?
7. ¿Por qué el speedup obtenido **no es igual** al número de threads disponibles? ¿Qué factores lo limitan?

1. Una operación es I/O-bound cuando su rendimiento está limitado por la velocidad de entrada/salida, como leer o escribir en disco o en la red. En cambio, una operación es CPU-bound cuando su rendimiento está limitado por la capacidad de procesamiento del CPU. La lectura de archivos desde disco es una operación I/O-bound, ya que el tiempo que tarda en completarse depende principalmente de la velocidad del disco y no del procesamiento del CPU. Esto se notaría al usar por ejemplo una SSD rápida vs. un disco duro tradicional, donde la lectura sería más rápida en el primer caso debido a la menor latencia y mayor velocidad de transferencia.
2. El GIL (Global Interpreter Lock) es un mecanismo de CPython que asegura que solo un hilo de ejecución pueda ejecutar código Python a la vez. Existe para proteger la integridad de los objetos internos de Python, evitando condiciones de carrera y corrupción de memoria. Sin embargo, el GIL introduce una limitación importante: impide que los hilos aprovechen múltiples núcleos de CPU para ejecutar código Python en paralelo, lo que puede limitar el rendimiento en operaciones CPU-bound.
3. Usamos Python a pesar del GIL porque es un lenguaje de alto nivel con una sintaxis sencilla y una gran cantidad de librerías para ciencia de datos, machine learning, visualización, etc. Además, Python actúa como un lenguaje de *pegamento* que permite integrar librerías de alto rendimiento escritas en otros lenguajes (como C, C++, Rust) que no tienen GIL, como NumPy, Arrow o PyTorch. Esto nos permite aprovechar el rendimiento de estas librerías mientras mantenemos la facilidad de uso y flexibilidad de Python.
4. `ThreadPoolExecutor` es adecuado para operaciones I/O-bound, ya que permite que otros hilos continúen ejecutándose mientras uno está esperando por una operación de entrada/salida. En cambio, `ProcessPoolExecutor` es más adecuado para operaciones CPU-bound, ya que cada proceso tiene su propio GIL, lo que permite aprovechar múltiples núcleos de CPU para ejecutar código Python en paralelo. Si la operación fuera puramente CPU-bound, usaría `ProcessPoolExecutor` para maximizar el rendimiento.
5. Crear un pool de threads introduce un overhead debido a la creación y gestión de los hilos, así como a la necesidad de sincronización entre ellos. Si los archivos fueran muy pequeños (1Kb como se dice en la pregunta), el overhead de crear y gestionar los threads podría superar el tiempo que se tarda en leer los archivos, lo que resultaría en un rendimiento peor que la lectura en serie.
6. Sí, se observó una mejora con la lectura paralela, especialmente a medida que aumenta el número de archivos. A partir de aproximadamente 12 archivos, la mejora comienza a ser notable, y se vuelve más significativa a medida que se agregan más archivos, aunque el speedup no es lineal debido al overhead y a las limitaciones del sistema.
7. El speedup obtenido no es igual al número de threads disponibles debido a varios factores, como el overhead de crear y gestionar los threads, la contención del GIL en operaciones que involucran código Python, la velocidad del disco y la cantidad de datos que se están leyendo. Además, si el sistema tiene un número limitado de núcleos de CPU o si otros procesos están utilizando recursos, esto también puede limitar el rendimiento y evitar que se alcance un speedup lineal.

# 2. Predicción de Valencia

La columna `valence` de Spotify mide el **positivismo musical** de una canción: valores cercanos a 1 indican canciones alegres y eufóricas, mientras que valores cercanos a 0 corresponden a canciones tristes o melancólicas. En esta sección analizaremos distintas formas de realizar predicciones con un modelo de regresión lineal ya entrenado, y luego entrenaremos un modelo más complejo.

## 2.1 Regresión Lineal a Mano [1.5 Puntos]

Antes de entrenar un modelo completo, veremos cómo **la elección de implementación** afecta drásticamente el rendimiento de predicción. Usaremos un modelo de regresión lineal pre-entrenado cuyos coeficientes ya están dados, e implementaremos la predicción usando cinco enfoques distintos: Python puro, NumPy, Numba (JIT), pandas y Polars.

**Código dado — carga de datos y parámetros del modelo:**

In [15]:
# Carga de datos y preparación del split
df_train = load_all_serial(DATA_DIR, n_batches=20)

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

X = df_train[PARAM_COLS + ["key", "mode", "genre"]]
y = df_train["valence"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
# Parámetros del modelo lineal pre-entrenado (dados)
params = {
    "danceability": 0.7718203106411208,
    "energy": 0.4252134896942928,
    "loudness": -0.008319917439312445,
    "speechiness": -0.24543273088867107,
    "acousticness": 0.10440236785191129,
    "instrumentalness": -0.11203723673701874,
    "liveness": 0.023790522969698424,
    "tempo": 0.0007885690378158087,
    "duration_ms": -4.31739613602265e-07,
    "year": -0.0036043842721972985,
}
intercept = 6.948154825159983

params_vals = list(params.values())
params_arr = np.array(params_vals, dtype=np.float32)

**Código dado — las 5 implementaciones de predicción:**

Analiza cómo cada implementación aborda el mismo problema y presta atención a las diferencias en legibilidad, concisión y (como verás en el benchmark) rendimiento.

In [17]:
def linear_regression_predict(X: np.ndarray, params: list[float]) -> list[float]:
    """Predicción con loop Python puro."""
    preds = []
    for row in X:
        val = intercept
        for j, w in enumerate(params):
            val += row[j] * w
        preds.append(val)
    return preds


def linear_regression_predict_numpy(X: np.ndarray, params: list[float]) -> np.ndarray:
    """Predicción vectorizada con NumPy."""
    return (X * np.array(params)).sum(axis=1) + intercept


@numba.njit
def linear_regression_predict_numba(X: np.ndarray, params: np.ndarray) -> np.ndarray:
    """Predicción con Numba JIT (loop compilado a código máquina)."""
    n = X.shape[0]
    preds = np.empty(n)
    for i in range(n):
        val = intercept
        for j in range(len(params)):
            val += X[i, j] * params[j]
        preds[i] = val
    return preds


def linear_regression_predict_pandas(X: pd.DataFrame, params: list[float]) -> pd.Series:
    """Predicción vectorizada con pandas (dot product)."""
    return X.dot(pd.Series(params, index=X.columns)) + intercept


def linear_regression_predict_polars(X: pl.DataFrame, params: list[float]) -> pl.Series:
    """Predicción vectorizada con Polars (expresiones lazy)."""
    weights = dict(zip(X.columns, params))
    expr = pl.lit(intercept)
    for col, w in weights.items():
        expr = expr + pl.col(col) * w
    return X.select(expr.alias("pred"))["pred"]

**Código dado — Benchmark de las 5 implementaciones:**

In [18]:
@dataclass
class TimeMeasurement:
    time_took: float
    iteration: int
    version: str


time_measurements: list[TimeMeasurement] = []

ranges = [10, 50, 100, 250, 500, 750, 1000, *range(1001, len(X_test) + 1, 1000)]

for it in ranges:
    X_np = X_test[PARAM_COLS].iloc[:it].to_numpy(dtype=np.float32)
    X_pd = X_test[PARAM_COLS].iloc[:it]
    X_pl = pl.from_pandas(X_pd)

    for name, fn, args in [
        ("Python", linear_regression_predict, (X_np, params_vals)),
        ("NumPy", linear_regression_predict_numpy, (X_np, params_vals)),
        ("Numba-JIT", linear_regression_predict_numba, (X_np, params_arr)),
        ("Pandas", linear_regression_predict_pandas, (X_pd, params_vals)),
        ("Polars", linear_regression_predict_polars, (X_pl, params_vals)),
    ]:
        t0 = time.perf_counter()
        fn(*args)
        time_measurements.append(TimeMeasurement(time.perf_counter() - t0, it, name))

df_bench = pd.DataFrame(time_measurements)

# Gráfico 1: tiempos absolutos
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
).show()

# Gráfico 2: tiempos absolutos (en log)
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación (en escala logarítmica)",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
    log_y=True,
).show()

# Gráfico 3: speedup relativo respecto a Python puro
pivot = df_bench.pivot(index="iteration", columns="version", values="time_took")
for col in ["NumPy", "Numba-JIT", "Pandas", "Polars"]:
    pivot[col] = pivot["Python"] / pivot[col]
pivot["Python"] = 1.0

melted = pivot.reset_index().melt(
    id_vars=["iteration"],
    value_vars=["Python", "NumPy", "Numba-JIT", "Pandas", "Polars"],
    value_name="speedup",
)
px.line(
    melted,
    x="iteration",
    y="speedup",
    color="version",
    markers=True,
    title="Speedup relativo respecto a Python puro",
    labels={"iteration": "Número de filas", "speedup": "Speedup (×)"},
).show()

### Preguntas [1.5 Puntos]

  1. ¿Qué es la vectorización en NumPy? ¿Cómo puede ejecutar operaciones sobre arrays sin loops de Python explícitos?
  2. ¿Qué es JIT (Just-In-Time compilation)? ¿Qué hace el decorador @numba.njit? ¿Qué significa el modo nopython?
  3. ¿Por qué Numba es más lento en la primera ejecución? ¿Qué es el warm-up de JIT y cómo lo manejamos en el benchmark?
  4. ¿Qué es Polars y cuáles son sus principales características como librería de datos? ¿Para qué escenarios fue diseñada y por qué ha ganado popularidad como alternativa a pandas?
  5. ¿En qué se diferencia Polars de pandas a nivel de implementación (lenguaje, modelo de ejecución, manejo de memoria)?
  6. ¿Por qué pandas puede ser más lento que NumPy aun usando operaciones vectorizadas internamente?
  7. ¿Qué son las instrucciones SIMD (Single Instruction Multiple Data)? ¿Cómo contribuyen a la aceleración de NumPy y Polars?
  8. ¿Cuándo conviene usar Numba sobre NumPy? ¿Y Polars sobre pandas para operaciones numéricas?
  9. ¿Cuál implementación fue la más rápida en tu medición? ¿Era esperable ese resultado?
  10. ¿Se observa diferencia notable entre pandas y NumPy? ¿Por qué pandas puede ser más lento o más rápido?
  11. ¿A partir de cuántas filas empieza a ser evidente la ventaja de NumPy/Numba sobre Python puro?
  12. ¿Polars fue más eficiente que pandas en tu medición? Verifica la versión de pandas instalada (pd.__version__) y comenta si crees que la versión influye en el resultado.
  13. ¿Por qué Numba puede igualar o superar a NumPy para loops numéricos simples?
  14. El benchmark excluye el costo de convertir datos a NumPy/Polars (la conversión ocurre fuera del timing). ¿Cómo cambiaría el resultado si incluyeras ese costo? ¿En qué escenarios de
  producción ese costo no existiría?
  15.  Si tuvieras que realizar esta predicción sobre 100 millones de filas en un servidor de producción, ¿qué implementación elegirías y por qué? ¿Cambiaría tu respuesta si dispusieras de
  una GPU?

**Escribe tus respuestas aquí...**

1- Es una técnica que permite aplicar operaciones matemáticas de matrices sobre un arreglo completo  de un solo golpe y directamente a nivel de hardware, eliminando los bucles for de Python. Esto es posible debido a que Numpy obliga a que todos los datos del arreglo sean del mismo tipo y los guarda alineados en un bloque de memoria contiguo, permitiendo que la CPU procese este bloque compacto de números de forma masiva, paralela y nativa. Para ejecutar operaciones sobre arrays sin loops de python, Numpy delega el control directamente al código complicado escrito en lenguaje C que corre por debajo de Numpy, haciendo que en vez de iterar fila por fila que hace Python, la CPU aplique instrucciones SIMD Single Instruction, Multiple Data. Esto permite ejecutar una única orden matemática sobre un lote masivo de números en paralelo y a nivel de hardware, eliminando por completo el tiempo perdido en verificar tipos de datos en cada paso.

2- Jit es una estrategia de ejecución que compila el código a lenguaje de máquina directamente optimizado. El decorador @numba.njit activa JIT en su modo estricto y rapido, de manera que la función que este luego del decorador se ejecutará a código máquina nativo. El modo nopython significa que la función compilada debe ejecutarse sin la intervención de Python, es decir variables, tipos de datos y operaciones se deben traducir directamnete a tipos nativos de C.

3- Numba es más lento en la primera ejecución debido al proceso warm-up de JIT, es decir la primera vez que se ejecuta tiene que analizar, traducir y compilar el código, por lo que en la primera ejecución se paga una penalización de tiempo, pero en las siguientes ejecuciones no ocurre porque el código binario ya quedó guardado en la memoria caché. En el benchmark se incluye el warm-up implícita en la primera iteración del bucle, por esto mismo en la primera iteración del benchmark con 10 número de filas el timpo de numba es la más alta pero luego disminuye.

4- Polars es una librería de manipulación de datos en formato DataFrame escrita en Rust, diseñada para ser veloz y eficiente al compilar sin el recolector de basura. Sus principales características son:
-  Escrita en Rus: Tiene un control absoluto sobre la memoria y el hadware
-  Modelo en memoria con Apache Arrow: al utilizar Arrow como su arquitectura interna de datos, permite que la información se almacene en memoria de forma columnar y contigua, optimizando masivamente la caché de la CPU.
-  Evaluación Perezosa: Polas permite trabajar en modo Lazy, lo que permite constuir un plan de optimización lógico, eliminando pasos innecesarios y optimizando las consultas antes utilizar un dato en el disco.
Polar fue diseñado para escenarios de Big Data Local donde es necesario procesar datasets masivos que pueden saturar la memoria RAM y para análitica de alto rendimiento que se requiera ejecutar Pipelines de ingeniería de datos donde se quiere procesar, filtrar y agregar información con alta velocidad usando el 100% de la CPU.
Polar se ha ganado popularidad como alternativa de Pandas debido que permite:
- Paralelismo nativo multi-hilo: Pandas ejecuta sus operaciones en un solo núcleo,mientras que Polar, reparte el trabajo de forma automática en todos los núcleos de la CPU.
- Eficiencia en memoria: Como Polars no tiene índices, implementa políticas de copia bajo demanda y procesa datos en bloques, consumiendo menos RAM que pandas al tener que crear copias intermedias e índices.
- Optimización automática: Pandas procesa toda la matriz incluso si es ineficiente. De lo contrario Polars en modo Lazy optimiza las consultas eliminando pasos innecesarios y moviendo filtrados.

5- Pandas fue escrito hace más de una década pensando en la flexibilidad de la CPU en un solo núcleo,mientras que Polars fue reescrito desde cero para aprovechar las arquitecturas de hadware modernas. Polars e


6- Pandas puede ser más lento que Numpy debido a que añade múltiples capas de abstracción e infraestructura. Esto incluye el costo de la alineación automática por índices, la reconstrucción de bloques mediante el BlockManager, la validación de tipos de datos, el manejo de metadatos y el uso del tipo Object. Además, al invocar una operación en Pandas, la librería debe validar los argumentos de entrada, extraer los arreglos internos de NumPy para realizar los cálculos optimizados en C, y finalmente volver a empaquetar el resultado en una estructura de DataFrame o Serie, lo que genera una sobrecarga constante de procesamiento.

7- SIMD es la capacidad arquitectónica de los procesadores modernos que permite al hadware ejecutar una misma operación matemática sobre varios elementos de datos simultánemente en un solo ciclo de reloj. Es decir, permite ejecutar una operación matemática en la CPU en una única instrucción. Contribuye a la aceleración de Numpy y Polars porque ambas librerías almacenan sus datos de forma contigua y homogénea en la memoria (usando arreglos de C y Apache Arrow, respectivamente). Esto le permite al procesador alimentar sus registros vectoriales directamente con ráfagas continuas de números brutos, ejecutando los cálculos en paralelo a nivel de circuitos y multiplicando la velocidad de procesamiento de forma nativa.

8- Numba conviene sobre NumPy cuando tenemos algoritmos matemáticos complejos con bucles for que no se pueden vectorizar tradicionalmente, o cuando queremos evitar que NumPy cree matrices temporales masivas en la RAM. Polars conviene sobre Pandas cuando trabajamos con datasets gigantescos que saturan la memoria, o cuando queremos que los cálculos numéricos se ejecuten de forma paralela y nativa en todos los núcleos de la CPU aprovechando el modo Lazy para optimizar el plan de ejecución.

### 2.2 Entrenamiento y Comparación de `n_jobs` [0.5 Puntos]

Ahora entrenaremos un modelo más complejo: un **RandomForestRegressor** que usa las características de audio más una codificación del género musical para predecir `valence`. Compararemos el efecto de paralelizar el entrenamiento con el parámetro `n_jobs`.

**Código dado — pipeline encapsulado** (no modificar):

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor


def build_pipeline(n_jobs: int = 1) -> Pipeline:
    # En producción este pipeline usaría LGBMRegressor; aquí usamos RandomForest
    # para ilustrar el efecto de n_jobs de forma más pronunciada.
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        (
                            "numerical",
                            "passthrough",
                            PARAM_COLS,
                        ),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

In [ ]:
# Entrena con n_jobs=1 y mide el tiempo
pipeline_1 = build_pipeline(n_jobs=1)
t0 = time.perf_counter()
pipeline_1.fit(X_train, y_train)
time_1job = time.perf_counter() - t0

# Entrena con n_jobs=-1 y mide el tiempo
pipeline_all = build_pipeline(n_jobs=-1)
t0 = time.perf_counter()
pipeline_all.fit(X_train, y_train)
time_all_jobs = time.perf_counter() - t0

# Calcula RMSE de ambos modelos
rmse_1 = root_mean_squared_error(y_test, pipeline_1.predict(X_test))
rmse_all = root_mean_squared_error(y_test, pipeline_all.predict(X_test))

print(f"n_jobs=1  → tiempo: {time_1job:.1f}s | RMSE: {rmse_1:.4f}")
print(f"n_jobs=-1 → tiempo: {time_all_jobs:.1f}s | RMSE: {rmse_all:.4f}")

In [ ]:
# Gráficos de tiempos y RMSE
df_perf = pd.DataFrame(
    {
        "configuracion": ["n_jobs=1", "n_jobs=-1"],
        "tiempo_s": [time_1job, time_all_jobs],
        "rmse": [rmse_1, rmse_all],
    }
)

px.bar(
    df_perf,
    x="configuracion",
    y="tiempo_s",
    title="Tiempo de entrenamiento según n_jobs",
    labels={"tiempo_s": "Tiempo (s)", "configuracion": "Configuración"},
    text_auto=".1f",
).show()

px.bar(
    df_perf,
    x="configuracion",
    y="rmse",
    title="RMSE según n_jobs",
    labels={"rmse": "RMSE", "configuracion": "Configuración"},
    text_auto=".4f",
).show()

### Preguntas [0.5 Puntos]

1. ¿Qué hace el parámetro `n_jobs` en RandomForest (y en general en scikit-learn)?
2. **¿Por qué aquí sí funciona el paralelismo real sin el problema del GIL?** (Pista: RandomForest en scikit-learn usa joblib con backend de procesos o threads nativos.)
3. ¿Cuánto mejoró el tiempo con `n_jobs=-1`? 
4. ¿Fue proporcional al número de CPUs disponibles en tu máquina? ¿Por qué no?
5. ¿Hubo diferencia en RMSE entre ambas versiones? ¿Era esperable? ¿Por qué?

**Escribe tus respuestas aquí...**

# 3. Orquestación del Pipeline con Apache Airflow

En producción, los pipelines de datos y ML rara vez se ejecutan a mano desde un notebook. Se necesita:
- **Automatización**: que el pipeline corra periódicamente (diariamente, por hora…).
- **Dependencias**: que el entrenamiento solo comience si la carga de datos terminó exitosamente.
- **Monitoreo y reintentos**: que si una tarea falla, el sistema lo registre y reintente.

**Apache Airflow** resuelve exactamente esto. Define pipelines como **DAGs** (*Directed Acyclic Graphs*), donde cada nodo es una **tarea** y las aristas definen dependencias.

| Concepto | Descripción |
|----------|-------------|
| **DAG** | Grafo Dirigido Acíclico que representa el pipeline completo |
| **Operator** | Unidad de trabajo (`PythonOperator`, `BashOperator`, …) |
| **Task** | Instancia de un Operator dentro de un DAG |
| **XCom** | Mecanismo para pasar datos pequeños entre tareas |
| **schedule** | Expresión cron que indica cuándo ejecutar el DAG |

### Setup local


En la carpeta del Lab:

```bash
export AIRFLOW_HOME=$(pwd)
airflow db migrate          # inicializa la base de datos de metadata
# Ver la contraseña. Si no se en un comienzo, ejecutar airflow standalone, parar el proceso y luego ejecutar nuevamente este comando. 
cat $AIRFLOW_HOME/simple_auth_manager_passwords.json.generated 
airflow standalone       # levanta scheduler + webserver en http://localhost:8080
```

Los DAGs deben guardarse en `./dags`.

## 3.1 Implementación del DAG

**TO-DO [0.8 Puntos]:**
- [ ] Implementar `task_load_data_fn`: cargar 5 batches en paralelo, guardar en disco como Parquet y pasar la ruta a la siguiente tarea usando XCom.
- [ ] Implementar `task_train_model_fn`: recuperar la ruta de XCom, cargar el DataFrame, preparar X e y, entrenar `build_pipeline(n_jobs=-1)` e imprimir el tiempo.
- [ ] Definir la dependencia entre tareas (`load_data >> train_model`).

El siguiente bloque es el template que debes completar en tu celda de respuesta.

In [ ]:
%%writefile ~/airflow/dags/spotify_pipeline_dag.py

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


DATA_DIR = Path("/RUTA/ABSOLUTA/A/Labs/Lab9_v2/data")  # AJUSTA esta ruta
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]


# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────


def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )


# ── Funciones de las tareas de Airflow ───────────────────────────────────────


def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    TODO: implementa esta función.
    - Usa load_all_parallel para cargar los datos.
    - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
    - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
    """
    ...


def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    TODO: implementa esta función.
    - Recupera la ruta del archivo desde XCom.
    - Lee el DataFrame desde esa ruta.
    - Prepara X e y, realiza el split 80/20.
    - Entrena build_pipeline(n_jobs=-1).
    - Imprime el tiempo de entrenamiento.
    """
    ...


# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    # TODO: define la dependencia entre tareas (load_data debe ejecutarse antes que train_model)
    ...


In [ ]:
# Escribe aquí tu código (copia el template y completa los TODOs)


Una vez guardado el archivo, ejecuta el DAG con:

```bash
airflow standalone
```


### Pega aquí el output de las steps del DAG

- Step 1: 
...


- Step 2:
...

### Preguntas [1.2 Puntos]

1. ¿Qué es un **DAG**? ¿Qué significa que sea *Directed* (dirigido) y *Acyclic* (acíclico)? ¿Por qué importa la propiedad acíclica en un pipeline de datos?
2. ¿Qué es **Apache Airflow**? ¿Para qué tipo de problemas está diseñado y cuál es su unidad mínima de trabajo?
3. ¿Qué son los **Operators**? ¿Qué diferencia hay entre `PythonOperator` y `BashOperator`? ¿Cuándo usarías cada uno?
4. ¿Qué es **XCom** en Airflow? ¿Cómo funciona internamente (¿dónde se almacena?)? ¿Por qué **no** es adecuado para pasar DataFrames grandes entre tareas?
5. ¿Qué alternativa concreta usaste para pasar el DataFrame entre `load_data` y `train_model`? ¿Cuál sería la alternativa recomendada en producción (S3, GCS, DVC…)?
6. ¿Qué es el parámetro `schedule` de un DAG? ¿Cómo lo configurarías para que corra todos los días a las 3 AM?
7. ¿Qué diferencia hay entre Airflow y otras herramientas como **Prefect**, **Dagster**, **Luigi**, **Kubeflow**? ¿Cuál es la principal crítica que se le hace a Airflow?
8. ¿Por qué conviene orquestar el pipeline en Airflow en vez de simplemente ejecutar un script Python end-to-end?
9. ¿Qué pasa si `load_data` falla a mitad de camino? ¿Airflow reintenta automáticamente? ¿Cómo controlarías el número máximo de reintentos?
10. ¿Qué ventaja tiene que las tareas estén separadas (carga y entrenamiento) vs. una sola tarea monolítica, desde el punto de vista de debugging y eficiencia?
11. ¿Cómo podemos alertar si es que algún paso falla? ¿O si la pipeline se ejecuta correctamente?
12. En un pipeline de producción real, ¿qué otras tareas añadirías al DAG?

**Escribe tus respuestas aquí...**

# Conclusión

Eso ha sido todo para el lab de hoy. Recuerda que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda, no dudes en contactarnos por el foro de U-Cursos.

<p align="center">
  <img src="https://media.giphy.com/media/l0HlBO7eyXzSZkJri/giphy.gif" width="300">
</p>